# Walkthrough: Building Scalable Data Pipelines

**Web Scraping (BeautifulSoup + Selenium + XPath) & API Integration untuk E-Commerce**

Notebook ini adalah versi *walkthrough* dari materi di folder `steps/`. Cocok dijalankan
sel-per-sel sambil menjelaskan tiap konsep.

**Untuk siapa:** peserta yang baru belajar web scraping & data pipeline.

**Prasyarat:**
- Paham dasar Python (variabel, list, dict, loop, fungsi).
- Sudah `uv sync` di root project (lihat `README.md`).
- Browser Chrome/Chromium terpasang (untuk bagian Selenium).

**Setelah selesai, peserta bisa:**
- Memahami alur sebuah data pipeline sederhana.
- Scraping website statis dengan **BeautifulSoup**.
- Scraping website dinamis dengan **Selenium** + **XPath**.
- Mengambil data dari **API**.
- Membersihkan data dengan **pandas** dan menyimpannya ke **database**.


## Outline

0. Setup & import
1. Memahami Data Pipeline
2. BeautifulSoup — scrape 1 item
3. BeautifulSoup — 1 halaman → dictionary list
4. BeautifulSoup — beberapa halaman (pagination)
5. Selenium — website dinamis
6. XPath
7. API
8. Cleaning data (pandas)
9. Simpan ke database (SQLite)
10. Latihan + pitfalls

### Alur pipeline yang kita bangun

```
Scrape 1 item → Scrape 1 halaman → Dictionary List → Beberapa Halaman
   → Cleaning Data → Koneksi Database → Buat Tabel & Insert
```

### Sumber data

| Teknik                 | Website                          |
| ---------------------- | -------------------------------- |
| BeautifulSoup (statis) | https://books.toscrape.com       |
| Selenium (dinamis)     | https://quotes.toscrape.com/js   |
| API                    | https://dummyjson.com/products   |


In [1]:
# Setup: import semua library yang dipakai sepanjang walkthrough
import re
import sqlite3

import pandas as pd
import requests
from bs4 import BeautifulSoup

print("Library siap dipakai ✅")


Library siap dipakai ✅


## 1. Memahami Data Pipeline

**Data pipeline** adalah alur **otomatis** untuk memindahkan data dari **sumber** ke
**tujuan**, melalui proses yang **terstruktur** dan **dapat diulang** (repeatable).

Di notebook ini kita membangun pipeline: ambil data dari web → bersihkan → simpan ke database.

---

## 2. BeautifulSoup — Scrape 1 Item

`BeautifulSoup` adalah library untuk *parsing* HTML, sehingga kita bisa mengambil data
berdasarkan tag/atribut. Dua metode utama:

- `find()` → mengambil **satu** elemen
- `find_all()` → mengambil **beberapa** elemen

Kita mulai dari satu item (satu buku) di `books.toscrape.com`.


In [2]:
# Ambil HTML halaman, lalu parsing dengan BeautifulSoup
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=10)
response.raise_for_status()
response.encoding = "utf-8"  # agar simbol £ tampil benar

soup = BeautifulSoup(response.text, "html.parser")

# find() -> ambil SATU item pertama (<article class="product_pod">)
item = soup.find("article", class_="product_pod")

judul = item.find("h3").find("a")["title"]
harga = item.find("p", class_="price_color").text
rating = item.find("p", class_="star-rating")["class"][1]
stok = item.find("p", class_="instock availability").text.strip()

print("Judul :", judul)
print("Harga :", harga)
print("Rating:", rating)
print("Stok  :", stok)


Judul : A Light in the Attic
Harga : £51.77
Rating: Three
Stok  : In stock


## 3. Scrape 1 Halaman → Dictionary List

Sekarang kita ambil **semua** produk di satu halaman dengan `find_all()`, lalu simpan
tiap produk sebagai **dictionary**, dikumpulkan ke dalam **list**.


In [3]:
def scrape_halaman(url: str) -> list[dict]:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    produk = []
    for item in soup.find_all("article", class_="product_pod"):  # SEMUA buku
        produk.append(
            {
                "nama": item.find("h3").find("a")["title"],
                "harga": item.find("p", class_="price_color").text,
                "rating": item.find("p", class_="star-rating")["class"][1],
            }
        )
    return produk


data_satu_halaman = scrape_halaman("https://books.toscrape.com/")
print(f"Dapat {len(data_satu_halaman)} produk")
data_satu_halaman[:3]


Dapat 20 produk


[{'nama': 'A Light in the Attic', 'harga': '£51.77', 'rating': 'Three'},
 {'nama': 'Tipping the Velvet', 'harga': '£53.74', 'rating': 'One'},
 {'nama': 'Soumission', 'harga': '£50.10', 'rating': 'One'}]

## 4. Scrape Beberapa Halaman (Pagination)

Halaman ke-N ada di pola URL `https://books.toscrape.com/catalogue/page-{N}.html`.
Kita looping nomor halaman dan menggabungkan hasilnya jadi satu list besar.

In [4]:
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"


def scrape_banyak_halaman(jumlah_halaman: int = 3) -> list[dict]:
    semua = []
    for halaman in range(1, jumlah_halaman + 1):
        url = BASE_URL.format(halaman)
        print(f"-> scraping halaman {halaman}")
        semua.extend(scrape_halaman(url))
    return semua


data_mentah = scrape_banyak_halaman(jumlah_halaman=3)
print(f"Total {len(data_mentah)} produk dari beberapa halaman")
data_mentah[:3]

-> scraping halaman 1


-> scraping halaman 2


-> scraping halaman 3


Total 60 produk dari beberapa halaman


[{'nama': 'A Light in the Attic', 'harga': '£51.77', 'rating': 'Three'},
 {'nama': 'Tipping the Velvet', 'harga': '£53.74', 'rating': 'One'},
 {'nama': 'Soumission', 'harga': '£50.10', 'rating': 'One'}]

## 5. Selenium — Website Dinamis

Beberapa website membuat kontennya dengan **JavaScript**. Contohnya
`https://quotes.toscrape.com/js/` — kalau diambil dengan `requests`, isinya kosong.

`Selenium` mengontrol **browser sungguhan**, jadi JavaScript ikut dijalankan.

> Sel di bawah akan **membuka jendela Chrome** sebentar (default `headless=False` supaya
> kelihatan saat demo). Driver di-download otomatis oleh **Selenium Manager**, jadi tidak
> perlu install ChromeDriver manual. Set `headless=True` kalau tidak ingin jendela terbuka.

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def buat_driver(headless: bool = False) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)


driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")
    quotes = driver.find_elements(By.CLASS_NAME, "quote")
    print(f"Dapat {len(quotes)} quote dari website dinamis")
    for q in quotes[:3]:
        teks = q.find_element(By.CLASS_NAME, "text").text
        penulis = q.find_element(By.CLASS_NAME, "author").text
        print(f"- {teks}  -- {penulis}")
finally:
    driver.quit()  # selalu tutup browser

Dapat 10 quote dari website dinamis
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”  -- Albert Einstein
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”  -- J.K. Rowling
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”  -- Albert Einstein


## 6. XPath

**XPath** adalah bahasa query untuk memilih elemen di dalam DOM (HTML/XML).
Bisa dicoba di [xpath-playground](https://scrapinghub.github.io/xpath-playground/).

Beberapa contoh:

| XPath                          | Arti                                       |
| ------------------------------ | ------------------------------------------ |
| `//h1`                         | semua tag `<h1>`                           |
| `//p[1]`                       | tag `<p>` pertama                          |
| `//*[@id="first-name"]`        | elemen dengan `id="first-name"`            |
| `//p[@class="plot"]`           | `<p>` dengan class persis `plot`           |
| `//p[contains(@class,"plot")]` | `<p>` yang class-nya **mengandung** `plot` |

Di Selenium, kita pakai `By.XPATH` untuk mencari elemen.

In [6]:
driver = buat_driver()
try:
    driver.get("https://quotes.toscrape.com/js/")

    # XPath: semua <div class="quote">
    quotes = driver.find_elements(By.XPATH, '//div[@class="quote"]')
    print(f"Dapat {len(quotes)} quote via XPath")

    for q in quotes[:3]:
        # XPath relatif (diawali ".") -> dicari DI DALAM elemen quote ini
        teks = q.find_element(By.XPATH, './/span[@class="text"]').text
        penulis = q.find_element(By.XPATH, './/small[@class="author"]').text
        print(f"- {teks}  -- {penulis}")

    print("\nJudul halaman (//h1):", driver.find_element(By.XPATH, "//h1").text)
finally:
    driver.quit()

Dapat 10 quote via XPath
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”  -- Albert Einstein
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”  -- J.K. Rowling
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”  -- Albert Einstein

Judul halaman (//h1): Quotes to Scrape


## 7. API

**API** adalah jalur resmi untuk mengambil data terstruktur langsung dari sistem penyedia.
Datanya biasanya berbentuk **JSON**, stabil, dan lebih cepat dibanding scraping.

Kita pakai API e-commerce publik: `https://dummyjson.com/products`.

In [7]:
resp = requests.get(
    "https://dummyjson.com/products",
    params={"limit": 5, "select": "title,price,category,brand"},
    timeout=10,
)
resp.raise_for_status()

data_api = resp.json()  # ubah JSON jadi dict Python
produk_api = data_api["products"]
print(f"Dapat {len(produk_api)} produk dari API")
produk_api

Dapat 5 produk dari API


[{'id': 1,
  'title': 'Essence Mascara Lash Princess',
  'price': 9.99,
  'category': 'beauty',
  'brand': 'Essence'},
 {'id': 2,
  'title': 'Eyeshadow Palette with Mirror',
  'price': 19.99,
  'category': 'beauty',
  'brand': 'Glamour Beauty'},
 {'id': 3,
  'title': 'Powder Canister',
  'price': 14.99,
  'category': 'beauty',
  'brand': 'Velvet Touch'},
 {'id': 4,
  'title': 'Red Lipstick',
  'price': 12.99,
  'category': 'beauty',
  'brand': 'Chic Cosmetics'},
 {'id': 5,
  'title': 'Red Nail Polish',
  'price': 8.99,
  'category': 'beauty',
  'brand': 'Nail Couture'}]

## 8. Cleaning Data (pandas)

Data hasil scraping masih "kotor": harga berupa teks `"£51.77"`, rating berupa kata `"Three"`.
Kita bersihkan dengan **pandas** sebelum disimpan:

1. Masukkan ke `DataFrame`
2. Ubah harga `"£51.77"` → `51.77` (float)
3. Ubah rating `"Three"` → `3` (int)
4. Buang baris kosong & duplikat

In [8]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df = pd.DataFrame(data_mentah)

# harga: "£51.77" -> 51.77 (buang semua karakter selain angka & titik)
df["harga"] = df["harga"].apply(lambda x: re.sub(r"[^0-9.]", "", x)).astype(float)

# rating: teks -> angka
df["rating"] = df["rating"].map(RATING_MAP)

# buang baris kosong & duplikat
df = df.dropna().drop_duplicates().reset_index(drop=True)

print("Tipe data tiap kolom:")
print(df.dtypes)
df.head()

Tipe data tiap kolom:
nama          str
harga     float64
rating      int64
dtype: object


,nama,harga,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5


## 9. Simpan ke Database (SQLite)

Langkah terakhir: simpan data bersih ke database. Kita pakai **SQLite** (bawaan Python,
tidak perlu install server). Alurnya: **buat koneksi → buat tabel → insert data**.

> Versi **PostgreSQL** (`psycopg2`) ada di `steps/09_simpan_database.py` kalau ingin
> mendemokan database server.

In [9]:
DB_PATH = "walkthrough.db"

# 1) buat koneksi
conn = sqlite3.connect(DB_PATH)

# 2) buat tabel
conn.execute(
    """
    CREATE TABLE IF NOT EXISTS produk (
        id     INTEGER PRIMARY KEY AUTOINCREMENT,
        nama   TEXT NOT NULL,
        harga  REAL,
        rating INTEGER
    )
    """
)
conn.execute("DELETE FROM produk")  # bersihkan dulu biar tidak dobel saat re-run

# 3) insert data (pakai DataFrame hasil cleaning)
df.to_sql("produk", conn, if_exists="append", index=False)
conn.commit()

# cek hasil: baca kembali dari database
hasil = pd.read_sql("SELECT * FROM produk LIMIT 5", conn)
conn.close()
print(f"Tersimpan ke {DB_PATH}")
hasil

Tersimpan ke walkthrough.db


,id,nama,harga,rating
0,1,A Light in the Attic,51.77,3
1,2,Tipping the Velvet,53.74,1
2,3,Soumission,50.10,1
3,4,Sharp Objects,47.82,4
4,5,Sapiens: A Brief History of Humankind,54.23,5


## 10. Latihan

1. Ubah `scrape_banyak_halaman` agar mengambil **5 halaman**, lalu hitung **rata-rata harga**
   buku menggunakan pandas.
2. (Bonus) Tambahkan kolom **link** ke hasil scraping, lalu simpan ke tabel database.

Coba dulu sebelum melihat contoh jawaban di sel berikutnya.

In [10]:
# Contoh jawaban latihan 1
data_5 = scrape_banyak_halaman(jumlah_halaman=5)

df_5 = pd.DataFrame(data_5)
df_5["harga"] = df_5["harga"].apply(lambda x: re.sub(r"[^0-9.]", "", x)).astype(float)

print(f"Jumlah buku : {len(df_5)}")
print(f"Rata-rata harga: £{df_5['harga'].mean():.2f}")

-> scraping halaman 1


-> scraping halaman 2


-> scraping halaman 3


-> scraping halaman 4


-> scraping halaman 5


Jumlah buku : 100
Rata-rata harga: £34.56


## Pitfalls & Extensions

**Kesalahan umum:**
- Lupa cek `response.raise_for_status()` → diam-diam memproses halaman error.
- `find()` mengembalikan `None` kalau elemen tidak ada → akan error saat `.text`. Selalu pastikan strukturnya benar.
- Lupa `driver.quit()` → banyak proses browser nyangkut di memori.
- Scraping terlalu cepat / agresif → hormati situs (kasih jeda, baca `robots.txt`).

**Extensions:**
- Jalankan pipeline lengkap dari terminal: `uv run python pipeline.py --halaman 5`.
- Tambahkan `time.sleep()` antar request, atau pakai `WebDriverWait` untuk menunggu elemen muncul.
- Simpan ke **PostgreSQL** (lihat `steps/09_simpan_database.py`).